# Random Forest Training - Occupancy Prediction
Trains a random forest classifier to predict bus occupancy level (low, medium, high, very_high).
uses class weights to handle imbalance without discarding data.


In [1]:
# --- imports ---
import pandas as pd
import numpy as np
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

print("Libraries imported successfully")

Libraries imported successfully


In [23]:
# --- configuration ---

# paths to X and y files generated by the feature engineering notebook
# DATASET_PATH = "./data"
DATASET_PATH = "../data"
USE_LAGS = True
suffix   = "with_lags" if USE_LAGS else "no_lags"

# X_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_X.parquet"
# Y_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_y.pkl"

X_PATH = f"{DATASET_PATH}/sunt_2024_4months_{suffix}_X.parquet"
Y_PATH = f"{DATASET_PATH}/sunt_2024_4months_{suffix}_y.pkl"

# where to save the trained model and results
# MODEL_PATH = "./occupancy"
MODEL_PATH = "/Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy"

# train/test split
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# random forest hyperparameters
N_ESTIMATORS     = 100
MAX_DEPTH        = 15
MIN_SAMPLES_SPLIT = 20
MIN_SAMPLES_LEAF  = 10

print(f"X path : {X_PATH}")
print(f"y path : {Y_PATH}")
print(f"model  : {MODEL_PATH}")

X path : ../data/sunt_2024_4months_with_lags_X.parquet
y path : ../data/sunt_2024_4months_with_lags_y.pkl
model  : /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy


In [3]:
# --- load data ---

X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("Dataset loaded:")
print(f"  X shape  : {X.shape}")
print(f"  y shape  : {y.shape}")
print(f"  features : {list(X.columns)}")
print(f"\nTarget distribution:")
dist = y.value_counts(normalize=True).sort_index()
for cls, pct in dist.items():
    count = (y == cls).sum()
    bar = '█' * int(pct * 50)
    print(f"  {cls:12s}: {count:>10,} ({pct*100:5.2f}%) {bar}")

Dataset loaded:
  X shape  : (39265404, 14)
  y shape  : (39265404,)
  features : ['route_short_name', 'direction_id', 'pt_sequence', 'stop_id', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'loading_lag_1', 'loading_lag_2', 'trip_stage', 'time_of_day', 'loading_mean_route']

Target distribution:
  high        :  4,414,836 (11.24%) █████
  low         : 23,558,711 (60.00%) █████████████████████████████
  medium      : 10,142,754 (25.83%) ████████████
  very_high   :  1,149,103 ( 2.93%) █


In [4]:
# --- analyze class imbalance and compute class weights ---
# random forest accepts class_weight directly in its constructor,
# so no need to compute sample_weight separately like in xgboost

class_counts = y.value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")
print(f"Majority class  : {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class  : {class_counts.idxmin()} ({class_counts.min():,})")

classes = np.unique(y)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(zip(classes, class_weights_array))

print(f"\nClass weights:")
for cls, weight in sorted(class_weights.items()):
    print(f"  {cls:12s}: {weight:.4f}")

Imbalance ratio : 20.5:1
Majority class  : low (23,558,711)
Minority class  : very_high (1,149,103)

Class weights:
  high        : 2.2235
  low         : 0.4167
  medium      : 0.9678
  very_high   : 8.5426


In [5]:
# --- stratified train/test split ---
# stratify=y ensures class distribution is preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train set : {len(X_train):,} records")
print(f"Test set  : {len(X_test):,} records")

print(f"\nTrain distribution:")
for cls, pct in y_train.value_counts(normalize=True).sort_index().items():
    print(f"  {cls:12s}: {pct*100:.2f}%")

print(f"\nTest distribution:")
for cls, pct in y_test.value_counts(normalize=True).sort_index().items():
    print(f"  {cls:12s}: {pct*100:.2f}%")

Train set : 31,412,323 records
Test set  : 7,853,081 records

Train distribution:
  high        : 11.24%
  low         : 60.00%
  medium      : 25.83%
  very_high   : 2.93%

Test distribution:
  high        : 11.24%
  low         : 60.00%
  medium      : 25.83%
  very_high   : 2.93%


In [6]:
# --- train random forest model ---

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    # class_weight passed directly: rf handles weighting internally
    class_weight=class_weights,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("Random Forest configuration:")
print(f"  n_estimators      : {N_ESTIMATORS}")
print(f"  max_depth         : {MAX_DEPTH}")
print(f"  min_samples_split : {MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf  : {MIN_SAMPLES_LEAF}")
print(f"  class_weight      : balanced (custom)")
print("\nTraining...")

start_time = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed:.2f}s ({elapsed/60:.1f} min)")

Random Forest configuration:
  n_estimators      : 100
  max_depth         : 15
  min_samples_split : 20
  min_samples_leaf  : 10
  class_weight      : balanced (custom)

Training...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:  6.5min



Training completed in 1513.76s (25.2 min)


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 24.7min finished


In [7]:
# --- evaluate model ---
# primary metric: f1 macro (treats all classes equally regardless of size)
# secondary metric: balanced accuracy (average recall per class)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

metrics = {
    'accuracy'         : (accuracy_score(y_train, y_pred_train),          accuracy_score(y_test, y_pred_test)),
    'balanced accuracy': (balanced_accuracy_score(y_train, y_pred_train),  balanced_accuracy_score(y_test, y_pred_test)),
    'f1 macro'         : (f1_score(y_train, y_pred_train, average='macro'), f1_score(y_test, y_pred_test, average='macro')),
}

print(f"{'Metric':<22} {'Train':>10} {'Test':>10} {'Gap':>10}")
print('-' * 55)
for name, (train_val, test_val) in metrics.items():
    print(f"{name:<22} {train_val:>10.4f} {test_val:>10.4f} {train_val - test_val:>10.4f}")

gap = metrics['balanced accuracy'][0] - metrics['balanced accuracy'][1]
if gap > 0.1:
    print(f"\nWarning: high gap ({gap:.2%}) — possible overfitting")
elif gap > 0.05:
    print(f"\nModerate gap ({gap:.2%}) — acceptable")
else:
    print(f"\nLow gap ({gap:.2%}) — model generalizes well")

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:   18.9s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:   50.3s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    3.8s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:   12.5s finished


Metric                      Train       Test        Gap
-------------------------------------------------------
accuracy                   0.9351     0.9344     0.0007
balanced accuracy          0.9205     0.9190     0.0015
f1 macro                   0.9081     0.9067     0.0014

Low gap (0.15%) — model generalizes well


In [8]:
# --- classification report ---
print("Classification Report (test set):")
print(classification_report(y_test, y_pred_test))

Classification Report (test set):
              precision    recall  f1-score   support

        high       0.87      0.89      0.88    882967
         low       0.98      0.96      0.97   4711742
      medium       0.88      0.90      0.89   2028551
   very_high       0.85      0.93      0.89    229821

    accuracy                           0.93   7853081
   macro avg       0.90      0.92      0.91   7853081
weighted avg       0.94      0.93      0.93   7853081



In [9]:
# --- confusion matrix ---

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred_test, labels=labels)

print("Confusion Matrix (test set):")
print(f"\n{'':>12}", end='')
for label in labels:
    print(f"{label:>12}", end='')
print("  <- predicted")
print('-' * (12 + 12 * len(labels)))
for i, label in enumerate(labels):
    print(f"{label:>12}", end='')
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end='')
    print("  | actual")

print("\nPer-class accuracy:")
for i, label in enumerate(labels):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc     = correct / total if total > 0 else 0
    status  = 'OK' if acc > 0.6 else 'LOW' if acc > 0.4 else 'POOR'
    print(f"  {status:4s} {label:12s}: {acc:.2%} ({correct:,}/{total:,})")

Confusion Matrix (test set):

                    high         low      medium   very_high  <- predicted
------------------------------------------------------------
        high     784,714       1,966      61,642      34,645  | actual
         low       6,753   4,515,764     188,300         925  | actual
      medium      89,930     113,238   1,823,821       1,562  | actual
   very_high      15,446          94         608     213,673  | actual

Per-class accuracy:
  OK   high        : 88.87% (784,714/882,967)
  OK   low         : 95.84% (4,515,764/4,711,742)
  OK   medium      : 89.91% (1,823,821/2,028,551)
  OK   very_high   : 92.97% (213,673/229,821)


In [10]:
# --- feature importance ---

importances   = model.feature_importances_
feature_names = X.columns.tolist()
indices       = np.argsort(importances)[::-1]

print("Top features by importance:")
print('-' * 55)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = '█' * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f}  {bar}")

cumsum = 0
for i, idx in enumerate(indices):
    cumsum += importances[idx]
    if cumsum >= 0.8:
        print(f"\nTop {i+1} features explain 80% of total importance")
        break

Top features by importance:
-------------------------------------------------------
 1. loading_lag_1             0.5619  ████████████████████████████
 2. loading_lag_2             0.3695  ██████████████████
 3. hour                      0.0206  █
 4. loading_mean_route        0.0114  
 5. route_progression         0.0089  
 6. is_rush_hour              0.0053  
 7. time_of_day               0.0050  
 8. pt_sequence               0.0042  
 9. route_short_name          0.0027  
10. stop_id                   0.0026  
11. direction_id              0.0025  
12. trip_stage                0.0021  
13. day_of_week               0.0020  
14. is_weekend                0.0014  

Top 2 features explain 80% of total importance


In [12]:
import os

In [13]:
# --- compare with baseline and xgboost ---
# baseline: always predict the majority class (simplest possible model)
# xgboost results are loaded from file if available (saved by the xgboost training notebook)
# otherwise falls back to hardcoded values from the original run

majority_class = y_train.value_counts().idxmax()
baseline_acc   = (y_test == majority_class).mean()

XGB_RESULTS_PATH = f"{MODEL_PATH}/xgb_results_{suffix}.pkl"
if os.path.exists(XGB_RESULTS_PATH):
    with open(XGB_RESULTS_PATH, 'rb') as f:
        xgb_results = pickle.load(f)
    xgb_acc      = xgb_results['accuracy']
    xgb_bal_acc  = xgb_results['balanced_accuracy']
    xgb_f1_macro = xgb_results['f1_macro']
    print("XGBoost results loaded from file")
else:
    xgb_acc      = 0.6172
    xgb_bal_acc  = 0.5894
    xgb_f1_macro = 0.5031
    print("XGBoost results using hardcoded values (run TrainingXGBoost first to update)")

rf_acc      = metrics['accuracy'][1]
rf_bal_acc  = metrics['balanced accuracy'][1]
rf_f1_macro = metrics['f1 macro'][1]

print(f"\n{'Model':<20} {'Accuracy':>10} {'Bal. Acc':>10} {'F1 Macro':>10}")
print('-' * 55)
print(f"{'Baseline':<20} {baseline_acc:>10.4f} {'—':>10} {'—':>10}")
print(f"{'Random Forest':<20} {rf_acc:>10.4f} {rf_bal_acc:>10.4f} {rf_f1_macro:>10.4f}")
print(f"{'XGBoost':<20} {xgb_acc:>10.4f} {xgb_bal_acc:>10.4f} {xgb_f1_macro:>10.4f}")
print(f"\nRandom Forest vs XGBoost:")
print(f"  balanced accuracy : {(rf_bal_acc - xgb_bal_acc)*100:+.2f}%")
print(f"  f1 macro          : {(rf_f1_macro - xgb_f1_macro)*100:+.2f}%")

XGBoost results using hardcoded values (run TrainingXGBoost first to update)

Model                  Accuracy   Bal. Acc   F1 Macro
-------------------------------------------------------
Baseline                 0.6000          —          —
Random Forest            0.9344     0.9190     0.9067
XGBoost                  0.6172     0.5894     0.5031

Random Forest vs XGBoost:
  balanced accuracy : +32.96%
  f1 macro          : +40.36%


In [24]:
# --- save model and results to drive ---

model_file = f"{MODEL_PATH}/rf_occupancy_{suffix}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved      -> {model_file}")

features_file = f"{MODEL_PATH}/rf_feature_names_{suffix}.pkl"
with open(features_file, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"Feature names    -> {features_file}")

rf_results = {
    'accuracy'         : rf_acc,
    'balanced_accuracy': rf_bal_acc,
    'f1_macro'         : rf_f1_macro,
}
rf_results_file = f"{MODEL_PATH}/rf_results_{suffix}.pkl"
with open(rf_results_file, 'wb') as f:
    pickle.dump(rf_results, f)
print(f"RF results       -> {rf_results_file}")

Model saved      -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/rf_occupancy_with_lags.pkl
Feature names    -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/rf_feature_names_with_lags.pkl
RF results       -> /Users/florenciagonzalez/Documents/bus-prediction-system/ml/occupancy/rf_results_with_lags.pkl


In [ ]:
# --- push changes to github ---
!git add 03_trainingRandomForest.ipynb
!git commit -m "update: rf training with drive saving, consistent metrics and xgboost comparison"
!git push origin floppy

[main 07fdc94] update: rf training with drive saving, consistent metrics and xgboost comparison
 1 file changed, 1 insertion(+)
 create mode 100644 Occupancy_dataloader_SUNT_OD (1).ipynb
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.94 KiB | 389.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/victoriaeleonor/occupancy-prediction-capstone.git
   c79f24e..07fdc94  main -> main
